In [ ]:
!pip install -q -U langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 11.7 MB/s eta 0:00:00


In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass(
    "Enter your OpenAI API key: "
)

Enter your OpenAI API key: ··········


In [ ]:
import sqlite3
import re
import time

from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    model="gpt-5.6-luna",
    temperature=0
)

print("=" * 60)
print("NL2SQL WITH SQLITE + LANGCHAIN + OPENAI")
print("=" * 60)

NL2SQL WITH SQLITE + LANGCHAIN + OPENAI


In [ ]:
conn = sqlite3.connect("sales_nl2sql.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS sales (
    SaleID INTEGER PRIMARY KEY,
    ProductName TEXT,
    Category TEXT,
    City TEXT,
    Quantity INTEGER,
    Amount REAL,
    SaleDate TEXT,
    Status TEXT
)
""")

conn.commit()

print("Sales table created successfully.")

Sales table created successfully.


In [ ]:
sales_data = [
    (1, "Laptop", "Electronics", "Chennai", 2, 120000, "2026-01-10", "Completed"),
    (2, "Mobile", "Electronics", "Coimbatore", 3, 75000, "2026-01-12", "Completed"),
    (3, "Headphones", "Accessories", "Bengaluru", 5, 25000, "2026-01-15", "Completed"),
    (4, "Tablet", "Electronics", "Chennai", 2, 60000, "2026-01-20", "Cancelled"),
    (5, "Keyboard", "Accessories", "Madurai", 4, 12000, "2026-01-22", "Completed"),
    (6, "Monitor", "Electronics", "Coimbatore", 3, 45000, "2026-02-02", "Completed"),
    (7, "Mouse", "Accessories", "Chennai", 10, 10000, "2026-02-05", "Completed"),
    (8, "Printer", "Electronics", "Bengaluru", 2, 30000, "2026-02-10", "Completed"),
    (9, "Laptop", "Electronics", "Madurai", 1, 65000, "2026-02-15", "Completed"),
    (10, "Mobile", "Electronics", "Chennai", 4, 100000, "2026-02-20", "Completed")
]

In [ ]:
cursor.executemany("""
INSERT OR IGNORE INTO sales
(
    SaleID,
    ProductName,
    Category,
    City,
    Quantity,
    Amount,
    SaleDate,
    Status
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", sales_data)

conn.commit()

print("10 records inserted successfully.")

10 records inserted successfully.


In [ ]:
print("\nDATABASE RECORDS")
print("-" * 60)

cursor.execute("""
SELECT *
FROM sales
""")

rows = cursor.fetchall()

for row in rows:
    print(row)


DATABASE RECORDS
------------------------------------------------------------
(1, 'Laptop', 'Electronics', 'Chennai', 2, 120000.0, '2026-01-10', 'Completed')
(2, 'Mobile', 'Electronics', 'Coimbatore', 3, 75000.0, '2026-01-12', 'Completed')
(3, 'Headphones', 'Accessories', 'Bengaluru', 5, 25000.0, '2026-01-15', 'Completed')
(4, 'Tablet', 'Electronics', 'Chennai', 2, 60000.0, '2026-01-20', 'Cancelled')
(5, 'Keyboard', 'Accessories', 'Madurai', 4, 12000.0, '2026-01-22', 'Completed')
(6, 'Monitor', 'Electronics', 'Coimbatore', 3, 45000.0, '2026-02-02', 'Completed')
(7, 'Mouse', 'Accessories', 'Chennai', 10, 10000.0, '2026-02-05', 'Completed')
(8, 'Printer', 'Electronics', 'Bengaluru', 2, 30000.0, '2026-02-10', 'Completed')
(9, 'Laptop', 'Electronics', 'Madurai', 1, 65000.0, '2026-02-15', 'Completed')
(10, 'Mobile', 'Electronics', 'Chennai', 4, 100000.0, '2026-02-20', 'Completed')


In [ ]:
cursor.execute("""
PRAGMA table_info(sales)
""")

schema_rows = cursor.fetchall()

schema = """
TABLE: sales
COLUMNS:
"""

for row in schema_rows:
    column_name = row[1]
    data_type = row[2]
    schema += f"{column_name} {data_type}\n"

print("\nDATABASE SCHEMA")
print("-" * 60)
print(schema)


DATABASE SCHEMA
------------------------------------------------------------

TABLE: sales
COLUMNS:
SaleID INTEGER
ProductName TEXT
Category TEXT
City TEXT
Quantity INTEGER
Amount REAL
SaleDate TEXT
Status TEXT



In [ ]:
def validate_sql(sql):

    sql = sql.strip()
    sql_upper = sql.upper()

    # Must start with SELECT
    if not sql_upper.startswith("SELECT"):
        return False

    # Reject multiple SQL statements
    if ";" in sql[:-1]:
        return False

    dangerous_commands = [
        "INSERT",
        "UPDATE",
        "DELETE",
        "DROP",
        "ALTER",
        "CREATE",
        "REPLACE",
        "ATTACH",
        "DETACH"
    ]

    for command in dangerous_commands:
        if re.search(rf"\b{command}\b", sql_upper):
            return False

    return True

In [ ]:

def clean_sql(sql):

    sql = sql.strip()

    sql = re.sub(
        r"```sql",
        "",
        sql,
        flags=re.IGNORECASE
    )

    sql = re.sub(
        r"```",
        "",
        sql
    )

    return sql.strip()

In [ ]:
def generate_sql(question):

    prompt = f"""
You are an expert SQLite SQL generator.

Convert the user's natural-language question
into a SQLite SELECT query.

DATABASE SCHEMA:
{schema}

RULES:

1. Generate ONLY SELECT queries.
2. Use ONLY the sales table.
3. Use ONLY the columns provided in the schema.
4. Do NOT generate INSERT.
5. Do NOT generate UPDATE.
6. Do NOT generate DELETE.
7. Do NOT generate DROP.
8. Do NOT generate ALTER.
9. Do NOT generate CREATE.
10. Do NOT generate REPLACE.
11. Do NOT generate ATTACH.
12. Do NOT generate DETACH.
13. Use valid SQLite syntax.
14. Return ONLY the SQL query.
15. Do not provide explanations.
16. Do not use markdown.

USER QUESTION:
{question}
"""

    for attempt in range(3):

        try:

            print(
                f"\nTrying OpenAI through LangChain - Attempt {attempt + 1}"
            )

            response = llm.invoke(prompt)

            sql = response.content

            sql = clean_sql(sql)

            print("OpenAI response received.")

            return sql

        except Exception as e:

            print("Attempt failed:")
            print(e)

            if attempt < 2:
                print("Retrying in 3 seconds...")
                time.sleep(3)

    raise Exception(
        "OpenAI model failed after 3 attempts."
    )

In [ ]:
def execute_sql(sql):

    print("\nGENERATED SQL")
    print("-" * 60)
    print(sql)

    if not validate_sql(sql):
        print("\nUnsafe SQL rejected.")
        return

    try:

        cursor.execute(sql)

        results = cursor.fetchall()

        print("\nQUERY RESULT")
        print("-" * 60)

        if results:

            for row in results:
                print(row)

        else:
            print("No records found.")

    except sqlite3.Error as e:

        print("\nSQLite Error:")
        print(e)

In [ ]:
print("\n")
print("=" * 60)
print("NATURAL LANGUAGE QUESTION")
print("=" * 60)

question = input(
    "\nAsk a question about the sales data: "
)

print("\nYour question:")
print(question)



NATURAL LANGUAGE QUESTION

Ask a question about the sales data: which city has highest sales

Your question:
which city has highest sales


In [ ]:
try:

    generated_sql = generate_sql(question)

    execute_sql(generated_sql)

except Exception as e:

    print("\nOpenAI / LangChain Error:")
    print(e)


Trying OpenAI through LangChain - Attempt 1
Attempt failed:
Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
Retrying in 3 seconds...

Trying OpenAI through LangChain - Attempt 2
Attempt failed:
Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}
Retrying in 3 seconds...

Trying OpenAI through LangChain - Attempt 3
Attempt failed:
Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted